# Create Iketani Science and Technology Foundation Awards (池谷科学技術振興財団)

Creates the grants of the Iketani Science and Technology Foundation
(公益財団法人 池谷科学技術振興財団) from the foundation's own grants database.

**Prerequisite:** `scripts/local/iketani_to_s3.py` (uploads the parquet).

**Source:** https://www.iketani-zaidan.or.jp/program/grants_search.html ("過去助成の検索"),
queried per fiscal year exactly as the "助成実績の一覧" buttons on
https://www.iketani-zaidan.or.jp/program/nendo.html do (POST fl=search, nendo=<code>).
Every grant FY2005-FY2026 is listed with its grant number (助成No.), programme,
grantee, institution and project title. Per-year totals match the search page's own
count and are >= the nendo.html summary (FY2006/FY2008 list 7 and 2 more grants than
that summary, each with its own grant number).

**2,918 grants, FY2005-2026:** 2,521 single-year research grants (単年度研究助成),
163 FY2026 research grants (研究助成, new 2-year scheme), 234 international exchange
grants (国際交流等助成: -C conference dispatch, -D researcher invitation; kept as a
research-funder pathway, `funding_type = 'grant'`). 100% title, institution, grantee.

**Amounts:** NULL. The foundation publishes only programme maxima (research up to
JPY 3M over 2 years, exchange up to JPY 750k), not per-grant amounts; amount check
(6.7) waived.

**Field mapping:** `display_name` = project title (exchange grants: the meeting /
visit); `funder_scheme` = programme; `start_year` = fiscal year; dates NULL;
`lead_investigator` = grantee (family-first: first token = family), affiliation =
institution as printed.

**`funder_award_id` (2.1.1):** the published 助成No. (e.g. `0371001-A`), which is what
grantees cite: about 720 of the ~960 acknowledgement shell rows for this funder carry
that form (e.g. `0351028-A`), so they collapse onto these awards.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320325276`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 0382jhd52, doi 10.13039/501100008656)

**Priority:** `571` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.iketani_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/iketani/iketani_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(fiscal_year) as min_year, MAX(fiscal_year) as max_year
FROM openalex.awards.iketani_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.iketani_raw;

In [ ]:
%sql
SELECT fiscal_year, programme, COUNT(*) AS n
FROM openalex.awards.iketani_raw GROUP BY 1, 2 ORDER BY 1, 2;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320325276 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320325276;

## Step 2: Create Iketani Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.iketani_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320325276  -- Iketani Science and Technology Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Iketani Science and Technology Foundation grant: ', g.grantee, ' (', g.fiscal_year, ')')) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.programme LIKE '%国際交流%' THEN 'grant' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'iketani_grants_search' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.fiscal_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.iketani_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Published grant numbers `NNNNNNN-[ACD]` only (e.g. `0371001-A`).

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.iketani_awards
WHERE NOT funder_award_id RLIKE '^0[0-9]{6}-[ACD]$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.iketani_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'iketani_grants_search' AND priority = 571;

-- Priority 571: direct-from-funder ingest of the foundation's own grants database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    571 as priority
FROM openalex.awards.iketani_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.iketani_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.iketani_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.iketani_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.iketani_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.iketani_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.iketani_awards;

In [ ]:
%sql
-- 2.1.1: acknowledgement shells whose award id equals a published grant number (they collapse onto these awards).
SELECT r.provenance, COUNT(*) AS shells, COUNT(a.id) AS matching_published_numbers
FROM openalex.awards.openalex_awards_raw r
LEFT JOIN openalex.awards.iketani_awards a ON a.id = r.id
WHERE r.funder_id = 4320325276 AND r.priority < 3
GROUP BY 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'iketani_grants_search'
GROUP BY provenance, priority;